# ViT on CIFAR-10

Vision Transformer (ViT) treats an image as a sequence of patches and
feeds them through a standard Transformer encoder.

**Key adaptations for CIFAR-10 (32×32 images):**
- `patch_size=4` → (32÷4)² = 64 patches — enough for attention to work
- `embed_dim=256, depth=4` — fewer layers with higher per-layer capacity
- `dropout=0.1` — crucial regularisation for small-data ViT training
- Training from scratch (no pre-training)

In [ ]:
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Project imports
p = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if p not in sys.path:
    sys.path.insert(0, p)

from core.cv import ViT

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 0.1
LABEL_SMOOTHING = 0.1
NUM_CLASSES = 10

torch.manual_seed(42);

### Data — CIFAR-10 with basic augmentation

In [ ]:
DATA_ROOT = (
    Path.cwd().parent / "assets" if Path.cwd().name == "apps" else Path.cwd() / "assets"
)

transform_train = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)
transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)

train_set = datasets.CIFAR10(
    DATA_ROOT, train=True, download=False, transform=transform_train
)
test_set = datasets.CIFAR10(
    DATA_ROOT, train=False, download=False, transform=transform_test
)

train_loader = DataLoader(
    train_set, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
test_loader = DataLoader(test_set, BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
# Sample a few training images
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
classes = [
    "airplane",
    "auto",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].permute(1, 2, 0).numpy().clip(0, 1))
    ax.set_title(classes[labels[i]], fontsize=9)
    ax.axis("off")
fig.suptitle("CIFAR-10 samples (augmented)");

### Model — ViT-Tiny (CIFAR-optimised)

In [ ]:
model = ViT(
    img_size=32,
    patch_size=4,
    in_channels=3,
    num_classes=NUM_CLASSES,
    embed_dim=256,
    depth=4,
    n_heads=8,
    d_ff=1024,
    dropout=0.1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"ViT-Tiny: {total_params / 1e6:.2f} M parameters")
print(model)

In [ ]:
# ── Optimizer & Scheduler ─────────────────────────────────────────────
optimizer = optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999)
)


# Cosine annealing with a linear warmup (first 10 epochs)
# ViT pre-training needs longer warmup to stabilise QK projections
def warmup_cosine_lr(epoch):
    if epoch < 10:
        return (epoch + 1) / 10  # linear warmup
    t = (epoch - 10) / (EPOCHS - 10)
    return 0.5 * (1.0 + math.cos(math.pi * t))


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine_lr)

### Training loop

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y, label_smoothing=LABEL_SMOOTHING)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(-1) == y).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def eval(model, loader):
    model.eval()
    total, correct = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        correct += (logits.argmax(-1) == y).sum().item()
        total += x.size(0)
    return correct / total

In [ ]:
train_losses, test_accs = [], []
best_acc = 0.0
t0 = time.time()

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    test_acc = eval(model, test_loader)
    train_losses.append(train_loss)
    test_accs.append(test_acc)
    best_acc = max(best_acc, test_acc)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch + 1:2d} | loss {train_loss:.4f} | train acc {train_acc:.3f} | "
        f"test acc {test_acc:.3f} | lr {lr:.2e}"
    )

elapsed = time.time() - t0
print(f"\nDone in {elapsed / 60:.1f} min.  Best test acc: {best_acc:.3f}")

## Analysis: What did the model learn?

Three visual probes into the trained ViT:

1. **CLS attention** — which image patches does the model "look at" to classify?
2. **Position embedding** — what spatial structure did it learn?
3. **Head diversity** — do different attention heads specialise on different patterns?

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(train_losses)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train Loss")
axes[0].set_title("Training loss")
axes[1].plot(test_accs, label=f"Best {best_acc:.3f}")
axes[1].axhline(best_acc, c="gray", ls="--", alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Test Accuracy")
axes[1].set_title("Test accuracy")
axes[1].legend()
fig.suptitle(f"ViT-Tiny on CIFAR-10 ({total_params / 1e3:.0f}K params)");

In [ ]:
@torch.no_grad()
def forward_with_attention(model, x):
    """Forward pass returning logits + per-block, per-head attention weights."""
    model.eval()
    x = model.patch_embed(x)

    all_attn = []  # list of (N, n_heads, 65, 65)
    for block in model.blocks:
        # Self-attention with weights captured
        residual = x
        x = block.norm_1(x)
        x, attn_w = block.attn(
            x,
            x,
            x,
            need_weights=True,
            average_attn_weights=False,  # keep per-head
        )
        x = residual + x

        # FFN
        residual = x
        x = block.norm_2(x)
        x = block.ff(x)
        x = residual + x

        all_attn.append(attn_w)  # (N, n_heads, seq, seq)

    x = model.norm(x)
    logits = model.head(x[:, 0])
    return logits, all_attn


# Grab one test batch for all visualizations
test_imgs, test_labels = next(iter(test_loader))
test_imgs, test_labels = test_imgs[:8].to(DEVICE), test_labels[:8].to(DEVICE)
logits, attn_weights = forward_with_attention(model, test_imgs[:4])
preds = logits.argmax(-1)
print(f"Sample acc: {(preds == test_labels[:4]).sum()}/4")

### 1. [CLS] Attention Maps

Which image patches does the [CLS] token attend to?  Strong weights mean
that patch is influential for the classification decision.

Below: [CLS] attention averaged over all heads, for early (layer 0),
middle (layer 2), and late (layer 3) blocks, overlaid on the input image.


In [ ]:
P = 4  # patch_size
n_patches = 8  # img_size // P

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
layers_to_show = [0, 2, 3]  # early / middle / late (depth=4)

for row in range(4):
    # Original image
    img = test_imgs[row].cpu().permute(1, 2, 0)
    mean = torch.tensor([0.4914, 0.4822, 0.4465])
    std = torch.tensor([0.2470, 0.2435, 0.2616])
    img = img * std + mean

    for col, layer in enumerate(layers_to_show):
        ax = axes[row][col]
        # Average [CLS] attention over all heads -> patches only
        attn = attn_weights[layer][row, :, 0, 1:]  # (n_heads, n_patches)
        attn = attn.mean(0).reshape(n_patches, n_patches).cpu()

        ax.imshow(img.clamp(0, 1))
        im = ax.imshow(attn, cmap="jet", alpha=0.6, extent=(0, 32, 32, 0))
        ax.set_title(f"Layer {layer}", fontsize=10)
        ax.axis("off")

    # Predicted class
    color = "green" if preds[row] == test_labels[row] else "red"
    axes[row][3].text(
        0.5,
        0.5,
        f"pred: {classes[preds[row]]}\ntrue: {classes[test_labels[row]]}",
        transform=axes[row][3].transAxes,
        ha="center",
        va="center",
        fontsize=10,
        color=color,
    )
    axes[row][3].axis("off")

fig.suptitle("[CLS] Attention over layers (heads averaged)", fontsize=13)
plt.tight_layout()

### 2. Position Embedding Structure

ViT adds a learnable position embedding to each patch.  If the model
has learned spatial relationships, nearby patches should have similar
embeddings.

**Left**: cosine similarity matrix of position embeddings (65×65).
The [CLS] token is index 0; the 64 patches follow in row-major order.

**Right**: PCA projection of the 64 patch position embeddings to 2D,
coloured by spatial row (top→bottom = red→blue).

In [ ]:
pos = model.patch_embed.pos_embed[0].detach()  # (65, 192)

# ── Cosine similarity matrix ───────────────────────────────────
pos_norm = F.normalize(pos, dim=-1)
sim = (pos_norm @ pos_norm.T).cpu()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

ax = axes[0]
im = ax.imshow(sim, cmap="RdBu_r", vmin=-1, vmax=1)
ax.axhline(0.5, color="gray", ls="--", lw=0.5)  # separator: CLS | patches
ax.axvline(0.5, color="gray", ls="--", lw=0.5)
ax.set_title("Position embedding cosine similarity")
ax.set_xlabel("Position index")
ax.set_ylabel("Position index")
fig.colorbar(im, ax=ax, shrink=0.8)

# ── PCA via SVD ────────────────────────────────────────────────
patch_pos = pos[1:].cpu()  # (64, 192), exclude CLS
centered = patch_pos - patch_pos.mean(0)
U, S, Vt = torch.linalg.svd(centered.float(), full_matrices=False)
pos_2d = centered @ Vt[:2].T  # (64, 2) — Vt rows = right singular vectors

# Colour by spatial row (0=top row, 7=bottom row)
rows = torch.arange(8).repeat_interleave(8).numpy()
sc = axes[1].scatter(
    pos_2d[:, 0], pos_2d[:, 1], c=rows, cmap="viridis", s=30, alpha=0.8
)
axes[1].set_title("PCA of patch position embeddings")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
cbar = fig.colorbar(sc, ax=axes[1], shrink=0.8, ticks=range(8))
cbar.set_label("Spatial row (top→bottom)")
plt.tight_layout()

### 3. Head Diversity

The last layer's [CLS] attention, separated by head.  Each head
should focus on a different spatial pattern — this is how multi-head
attention enriches the model's representational capacity.

In [ ]:
n_heads = 8  # must match model.n_heads
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

img_idx = 0  # which test image to show
img = test_imgs[img_idx].cpu().permute(1, 2, 0)
img = img * std + mean

for h in range(n_heads):
    ax = axes.flat[h]
    # Last layer, head h, [CLS] -> patches
    attn = attn_weights[-1][img_idx, h, 0, 1:]  # (64,)
    attn = attn.reshape(n_patches, n_patches).cpu()

    ax.imshow(img.clamp(0, 1))
    ax.imshow(attn, cmap="jet", alpha=0.6, extent=(0, 32, 32, 0))
    ax.set_title(f"Head {h}", fontsize=10)
    ax.axis("off")

fig.suptitle(
    f"Last layer — per-head [CLS] attention\ntrue={classes[test_labels[img_idx]]}, pred={classes[preds[img_idx]]}",
    fontsize=11,
)
plt.tight_layout()

### Inspect predictions

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, n=12):
    model.eval()
    imgs, labels = next(iter(loader))
    imgs, labels = imgs[:n].to(DEVICE), labels[:n].to(DEVICE)
    logits = model(imgs)
    preds = logits.argmax(-1)
    fig, axes = plt.subplots(2, 6, figsize=(10, 4))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i].cpu().permute(1, 2, 0).numpy().clip(0, 1))
        color = "green" if preds[i] == labels[i] else "red"
        ax.set_title(
            f"true={classes[labels[i]]}\npred={classes[preds[i]]}",
            fontsize=8,
            color=color,
        )
        ax.axis("off")
    plt.tight_layout()


show_predictions(model, test_loader)

### Summary

**1. Position Embedding Topology**

Despite being initialised as a random 1D sequence, the learnable position embeddings self-organise into a clear $8 \times 8$ periodic structure in the cosine similarity matrix after 60 epochs. PCA reveals that PC1 and PC2 **perfectly decouple spatial rows and columns** respectively — the model has reconstructed a 2D grid geometry from a flat index, confirming a mature internal spatial localisation mechanism.

**2. Layer-wise Attention Progression**

[CLS] attention follows a clear local-to-global abstraction pattern: shallow layers (Layer 0) respond to high-contrast edges and local texture, while deeper layers (Layers 2–3) progressively converge on the object of interest and its contextual features. The $P=4$ patch size (64 patches) provides sufficient spatial resolution for $32 \times 32$ images, avoiding the spatial collapse that plagues $P=16$ at this resolution.

**3. Head Diversity & Functional Specialisation**

No head collapse observed. All 8 attention heads in the final layer exhibit distinct functional roles despite operating on only $d_k=24$ subspaces. Several heads specialise in bilaterally symmetric features (e.g. left/right ears, forehead/nose bridge), while others act as background or environmental anchors. This complementary division of labour explains the blocky structure seen in the mean attention heatmaps.